<a href="https://colab.research.google.com/github/sarahshahrir/ligand-docking/blob/experiment/DiffDock_Inf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone DiffDock


In [9]:
!git clone https://github.com/gcorso/DiffDock.git
%cd DiffDock

Cloning into 'DiffDock'...
remote: Enumerating objects: 520, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 520 (delta 209), reused 154 (delta 154), pack-reused 228 (from 3)
Receiving objects: 100% (520/520), 233.08 MiB | 32.69 MiB/s, done.
Resolving deltas: 100% (244/244), done.
/content/DiffDock/DiffDock


Install Dependencies

In [10]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.10.0+cu128
12.8


In [11]:
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q e3nn biopython spyrmsd
!pip install -q pandas scipy scikit-learn pyyaml tqdm networkx
!pip install -q rdkit
!pip install -q fair-esm
!pip install torch_cluster -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install -q prody
!pip install py3Dmol

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html


Check that GPU is available

In [12]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

import torch_geometric
import rdkit
print("torch_geometric OK")
print("rdkit OK")

True
Tesla T4
torch_geometric OK
rdkit OK


Run Inference On Single Complex

In [13]:
!pwd

/content/DiffDock/DiffDock


In [15]:
!python -m inference \
  --config default_inference_args.yaml  \
  --protein_ligand_csv data/protein_ligand_example.csv \
  --out_dir results/user_predictions_small

Generating ESM language model embeddings
Processing 1 of 1 batches (4 sequences)
0it [00:00, ?it/s]/content/DiffDock/DiffDock/datasets/parse_chi.py:91: RuntimeWarning: invalid value encountered in cast
  Y = indices.astype(int)
2it [01:08, 34.44s/it]


In [21]:
def visualize(filename):
    import py3Dmol

    view = py3Dmol.view(width=800, height=600)

    with open(f"data/{filename}/{filename}_protein_processed.pdb") as f:
        view.addModel(f.read(), "pdb")

    with open(f"results/user_predictions_small/{filename}/rank1.sdf") as f:
        view.addModel(f.read(), "sdf")

    view.addSurface(py3Dmol.VDW, {'opacity': 0.75, 'color': 'lightgray'}, {'model': 0})
    view.setStyle({'model': 1}, {'stick': {'radius': 0.2}})

    view.zoomTo({'model': 1})
    view.show()

In [ ]:
visualize("1a0q")

## IFP

Install ProLIF

In [22]:
pip install prolif

Load Protein + Ligand

In [31]:
import MDAnalysis as mda
import prolif as plf

# Paths
PROTEIN_PDB = "data/1a0q/1a0q_protein_processed.pdb"
POSE_SDF    = "results/user_predictions_small/1a0q/rank1.sdf"

# Load Protein
prot_u   = mda.Universe(PROTEIN_PDB)
prot_mol = plf.Molecule.from_mda(prot_u)

# Load Ligand Pose
from rdkit.Chem import SDMolSupplier

suppl   = SDMolSupplier(POSE_SDF, removeHs=False, sanitize=True)
mol     = next(m for m in suppl if m is not None)
lig_mol = plf.Molecule(mol)

# Run Fingerprint (Ask Kyle want which fingerprint!)
fp = plf.Fingerprint([
    "HBDonor", "HBAcceptor",
    "PiStacking", "CationPi",
    "Cationic", "Hydrophobic",
    "VdWContact"
])

# Here, we can pass in multiple poses in lig_mol: from rank 1-5.
# This allows the model to see multiple possible binding poses.
fp.run_from_iterable([lig_mol], prot_mol)

# Inspect Results
df = fp.to_dataframe()
print(df)

# See only active interactions (filter out False columns)
active = df.loc[:, df.any()]
print("\nActive interactions only:")
print(active)

# Save
df.to_csv("rank1_ifp.csv")

/usr/local/lib/python3.12/dist-packages/MDAnalysis/converters/RDKit.py:575: UserWarning: No `bonds` attribute in this AtomGroup. Guessing bonds based on atoms coordinates
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

ligand            UNK0                                                       
protein         SER7.L     PRO8.L    LYS18.L    THR20.L               THR22.L
interaction VdWContact VdWContact VdWContact HBAcceptor VdWContact VdWContact
Frame                                                                        
0                 True       True       True       True       True       True

Active interactions only:
ligand            UNK0                                                       
protein         SER7.L     PRO8.L    LYS18.L    THR20.L               THR22.L
interaction VdWContact VdWContact VdWContact HBAcceptor VdWContact VdWContact
Frame                                                                        
0                 True       True       True       True       True       True


## To Ask Kyle:
For PPAR receptors, which types of ligand–residue interactions are known to influence activation (e.g., hydrogen bonds, hydrophobic contacts, π-stacking, electrostatics)?